In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import json
import re
import shutil
import zipfile

import ezc3d
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from scipy.signal import butter, sosfiltfilt
from scipy.stats import spearmanr, t as student_t

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data' / 'raw' / 'hitting'
OUT = ROOT / 'results' / 'head_stability'
FIG = ROOT / 'figures' / 'head_stability'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True
CUTOFFS_HZ = (8, 12, 16)
PRIMARY_CUTOFF_HZ = 12
MIN_SWINGS = 5
BOOTSTRAPS = 1000
SEED = 20261001
MAX_GAP_FRAMES = 3
MATCH_RMSE_LIMIT_M = 0.005
HEAD_PAIR_CHANGE_LIMIT_M = 0.03
SELECTED_SESSION = None
BASE = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
CSV_BASE = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_hitting/data/'
ARCHIVES = {
    'hitting_c3d.zip': '083980649dfb4c3069a4f10baff57f156fcc4fb1d52decdba24f487e2ebc18cf',
    'hitting_landmarks.zip': '9cc2bdc94f5f5a6c5d540f4abe459caae2c5164cd53f6a6fd23f4b2b7283bcb3',
}
print('Head Movement, Repeatability & Batted-Ball Outcomes')
print('First run downloads about 498 MB if the C3D and landmark inputs are missing.')
print('Translation is a four-marker centroid proxy. No eye tracking or head-rotation inference.')

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def download(url, path):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {path}. Set AUTO_DOWNLOAD=True or supply the official file.')
    temporary = path.with_suffix(path.suffix + '.part')
    print(f'Downloading {path.name} ...', flush=True)
    urlretrieve(url, temporary)
    temporary.replace(path)

def ensure_archive(name, needed):
    if needed.exists():
        return
    archive = RAW / name
    if not archive.exists():
        download(BASE + name, archive)
    if file_sha256(archive) != ARCHIVES[name]:
        raise ValueError(f'{name} failed the Dataset v1 SHA-256 check. Remove it and retry.')
    with zipfile.ZipFile(archive) as zf:
        if name == 'hitting_c3d.zip':
            members = [n for n in zf.namelist() if n.lower().endswith('.c3d')]
            for member in members:
                relative = Path(member)
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError('Unsafe archive member')
                target = RAW / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as source, target.open('wb') as dest:
                    shutil.copyfileobj(source, dest)
        else:
            members = [n for n in zf.namelist() if Path(n).name == 'landmarks.csv']
            if len(members) != 1:
                raise ValueError('Expected one landmarks.csv')
            with zf.open(members[0]) as source, needed.open('wb') as dest:
                shutil.copyfileobj(source, dest)
    if not needed.exists():
        raise FileNotFoundError(f'Archive did not create {needed}')

ensure_archive('hitting_c3d.zip', RAW / 'c3d')
ensure_archive('hitting_landmarks.zip', RAW / 'landmarks.csv')
for local, remote in [('metadata.csv', 'metadata.csv'), ('poi_metrics.csv', 'poi/poi_metrics.csv'),
                       ('hittrax.csv', 'poi/hittrax.csv')]:
    if not (RAW / local).exists():
        download(CSV_BASE + remote, RAW / local)

metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_swing': str, 'user': str, 'session': str})
for key in ('user', 'session'):
    metadata[key] = metadata[key].map(lambda value: str(int(value)))
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
hittrax = pd.read_csv(RAW / 'hittrax.csv', dtype={'session_swing': str})
landmark_columns = ['session_swing', 'time', 'fp_100_time', 'contact_time'] + [
    side + 'wjc_' + axis for side in ('l', 'r') for axis in 'xyz']
landmarks = pd.read_csv(RAW / 'landmarks.csv', usecols=landmark_columns,
                        dtype={'session_swing': str})
for name, table in [('metadata', metadata), ('POI', poi), ('HitTrax', hittrax)]:
    if table.session_swing.duplicated().any():
        raise ValueError(f'Duplicate swing keys in {name}')
if landmarks.duplicated(['session_swing', 'time']).any():
    raise ValueError('Duplicate landmark time keys')
groups = {str(k): g.sort_values('time').reset_index(drop=True)
          for k, g in landmarks.groupby('session_swing', sort=False)}
print(f'{len(metadata)} processed swings, {metadata.user.nunique()} athletes')

In [ ]:
def read_trial(path):
    parts = path.stem.split('_')
    if len(parts) != 7 or parts[4] not in ('L', 'R'):
        return None
    c = ezc3d.c3d(str(path))
    labels = [str(x).strip().upper() for x in c['parameters']['POINT']['LABELS']['value']]
    required = ['LFHD', 'RFHD', 'LBHD', 'RBHD', 'LWRA', 'LWRB', 'RWRA', 'RWRB']
    if any(labels.count(x) != 1 for x in required):
        raise ValueError('missing or duplicate head/wrist marker labels')
    unit = str(c['parameters']['POINT']['UNITS']['value'][0]).strip().lower()
    if unit not in ('m', 'mm', 'cm'):
        raise ValueError(f'unknown point unit: {unit}')
    scale = {'m': 1., 'mm': .001, 'cm': .01}[unit]
    points = c['data']['points'][:3].transpose(2, 1, 0) * scale
    residual = c['data']['meta_points']['residuals'][0].T
    bad = (residual < 0) | (~np.isfinite(points).all(axis=2)) | (np.linalg.norm(points, axis=2) == 0)
    points[bad] = np.nan
    head = points[:, [labels.index(x) for x in required[:4]], :]
    wrists = np.stack([(points[:, labels.index(s + 'WRA')] + points[:, labels.index(s + 'WRB')]) / 2
                       for s in ('L', 'R')], axis=1)
    fs = float(c['header']['points']['frame_rate'])
    return dict(path=path, user=str(int(parts[0])), session=str(int(parts[1])), side=parts[4],
                original_swing=int(parts[5]), filename_ev=float(parts[6]) / 10,
                fs=fs, first_frame=int(c['header']['points']['first_frame']), head=head, wrists=wrists)

def match_trial(trial, candidates):
    scores = []
    for swing_id in candidates:
        g = groups.get(swing_id)
        if g is None or len(g) != len(trial['head']):
            continue
        t = g.time.to_numpy(float)
        if not np.allclose(t, np.arange(len(t)) / trial['fs'], atol=0.00006, rtol=0):
            continue
        target = np.stack([g[[s + 'wjc_' + a for a in 'xyz']].to_numpy(float)
                           for s in ('l', 'r')], axis=1)
        valid = np.isfinite(target).all(axis=(1, 2)) & np.isfinite(trial['wrists']).all(axis=(1, 2))
        if valid.mean() < .95:
            continue
        rmse = float(np.sqrt(np.mean((target[valid] - trial['wrists'][valid]) ** 2)))
        scores.append((rmse, swing_id))
    scores.sort()
    if not scores or scores[0][0] > MATCH_RMSE_LIMIT_M:
        raise ValueError('no frame/time/wrist-trajectory match')
    if len(scores) > 1 and scores[1][0] < max(.002, 3 * scores[0][0]):
        raise ValueError('ambiguous wrist-trajectory match')
    return scores[0][1], scores[0][0], scores[1][0] if len(scores) > 1 else np.nan

candidate_lookup = metadata.groupby(['user', 'session', 'hitter_side']).session_swing.apply(list).to_dict()
trial_by_swing, file_audit, duplicate_swing_ids = {}, [], set()
for path in sorted((RAW / 'c3d').rglob('*.c3d')):
    if len(path.stem.split('_')) != 7:
        file_audit.append(dict(file=path.name, status='static/model excluded'))
        continue
    try:
        trial = read_trial(path)
        if trial is None:
            raise ValueError('unrecognized swing filename')
        candidates = candidate_lookup.get((trial['user'], trial['session'], trial['side']), [])
        swing_id, error, runner_up = match_trial(trial, candidates)
        if swing_id in duplicate_swing_ids:
            raise ValueError(f'duplicate C3D match for {swing_id}')
        if swing_id in trial_by_swing:
            del trial_by_swing[swing_id]
            duplicate_swing_ids.add(swing_id)
            for previous in file_audit:
                if previous.get('session_swing') == swing_id:
                    previous['status'] = 'duplicate C3D match excluded'
            raise ValueError(f'duplicate C3D match for {swing_id}')
        trial_by_swing[swing_id] = trial
        file_audit.append(dict(file=path.name, session_swing=swing_id, status='matched',
                               original_swing=trial['original_swing'], wrist_rmse_m=error,
                               runner_up_rmse_m=runner_up, first_frame=trial['first_frame']))
    except (ValueError, RuntimeError, OSError) as exc:
        file_audit.append(dict(file=path.name, status=str(exc)))
file_audit = pd.DataFrame(file_audit)
file_audit.to_csv(OUT / 'c3d_match_audit.csv', index=False)
display(file_audit.status.value_counts().rename('files').to_frame())
print('Swing numbers in C3D filenames are not assumed to equal processed swing IDs.')

In [ ]:
def interpolate_short_gaps(values, max_gap=MAX_GAP_FRAMES):
    result = np.asarray(values, dtype=float).copy()
    missing = ~np.isfinite(result).all(axis=1)
    if missing.any():
        edges = np.diff(np.r_[False, missing, False].astype(int))
        starts, stops = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
        if any(a == 0 or b == len(result) or b - a > max_gap for a, b in zip(starts, stops)):
            raise ValueError('head gap exceeds limit or touches filter-window edge')
        valid = ~missing
        for axis in range(result.shape[1]):
            result[:, axis] = np.interp(np.arange(len(result)), np.flatnonzero(valid), result[valid, axis])
    return result, int(missing.sum())

def head_features(trial, g, height_m, side, cutoff):
    event_columns = ['fp_100_time', 'contact_time']
    if any(g[c].nunique(dropna=False) != 1 for c in event_columns):
        raise ValueError('inconsistent event annotations')
    plant, contact = g[event_columns].iloc[0].to_numpy(float)
    t = np.arange(len(trial['head'])) / trial['fs']
    if not np.isfinite([plant, contact, height_m]).all() or height_m <= 0:
        raise ValueError('missing event or height')
    if not (t[0] <= plant < contact <= t[-1]) or not .04 <= contact - plant <= .40:
        raise ValueError('invalid event range/order or plant-to-contact duration')
    start, stop = min(plant, contact - .10) - .20, contact + .10
    mask = (t >= start) & (t <= stop)
    tt = t[mask]
    if len(tt) < 40 or tt[0] > min(plant, contact - .10) - .05 or tt[-1] < contact + .05:
        raise ValueError('insufficient filter margin')
    marker_tracks, filled = [], 0
    for marker in range(4):
        track, count = interpolate_short_gaps(trial['head'][mask, marker])
        marker_tracks.append(track)
        filled += count
    head_markers = np.stack(marker_tracks, axis=1)
    pair_change = max(np.ptp(np.linalg.norm(head_markers[:, a] - head_markers[:, b], axis=1))
                      for a in range(4) for b in range(a + 1, 4))
    if pair_change > HEAD_PAIR_CHANGE_LIMIT_M:
        raise ValueError('head-marker pair distance change exceeds 30 mm')
    center = head_markers.mean(axis=1)
    smooth = sosfiltfilt(butter(4, cutoff, fs=trial['fs'], output='sos'), center, axis=0)
    at_plant = np.array([np.interp(plant, tt, smooth[:, a]) for a in range(3)])
    signs = np.array([1, 1 if side == 'R' else -1, 1])
    displacement_cm = (smooth - at_plant) * 100 * signs
    features = dict(plant_time=plant, contact_time=contact, plant_to_contact_ms=1000 * (contact - plant),
                    interpolated_marker_frames=filled, head_pair_change_mm=pair_change * 1000)
    for prefix, lo in [('plant', plant), ('late100', contact - .10)]:
        grid = np.r_[lo, tt[(tt > lo) & (tt < contact)], contact]
        xyz = np.column_stack([np.interp(grid, tt, displacement_cm[:, a]) for a in range(3)])
        for axis, label in enumerate(['forward', 'lateral', 'vertical']):
            features[f'{prefix}_{label}_range_cm'] = float(np.ptp(xyz[:, axis]))
            features[f'{prefix}_{label}_net_cm'] = float(xyz[-1, axis] - xyz[0, axis])
        features[f'{prefix}_max_disp_cm'] = float(np.linalg.norm(xyz - xyz[0], axis=1).max())
        features[f'{prefix}_max_disp_pct_height'] = features[f'{prefix}_max_disp_cm'] / height_m
        features[f'{prefix}_vertical_range_pct_height'] = features[f'{prefix}_vertical_range_cm'] / height_m
    normalized_t = np.linspace(plant, contact, 101)
    curve = np.column_stack([np.interp(normalized_t, tt, displacement_cm[:, a]) for a in range(3)])
    features.update({f'contact_{label}_cm': curve[-1, a] for a, label in enumerate(['forward', 'lateral', 'vertical'])})
    return features, curve, dict(time_ms=(tt - plant) * 1000, displacement_cm=displacement_cm)

meta_index = metadata.set_index('session_swing')
records, extraction_audit, curves, traces = [], [], {}, {}
for swing_id, meta in meta_index.iterrows():
    for cutoff in CUTOFFS_HZ:
        try:
            if swing_id not in trial_by_swing:
                raise ValueError('no validated C3D match')
            row, curve, trace = head_features(trial_by_swing[swing_id], groups[swing_id],
                                             float(meta.session_height_in) * .0254, meta.hitter_side, cutoff)
            row.update(session_swing=swing_id, cutoff_hz=cutoff, user=meta.user, session=meta.session,
                       hitter_side=meta.hitter_side, height_m=float(meta.session_height_in) * .0254)
            records.append(row)
            if cutoff == PRIMARY_CUTOFF_HZ:
                curves[swing_id], traces[swing_id] = curve, trace
            status = 'retained'
        except (ValueError, KeyError) as exc:
            status = str(exc)
        extraction_audit.append(dict(session_swing=swing_id, cutoff_hz=cutoff, status=status))
audit = pd.DataFrame(extraction_audit)
audit.to_csv(OUT / 'extraction_audit.csv', index=False)
display(audit[audit.cutoff_hz.eq(PRIMARY_CUTOFF_HZ)].status.value_counts().rename('swings').to_frame())
if not records:
    raise ValueError('No swings retained. Inspect c3d_match_audit.csv and extraction_audit.csv.')
all_features = pd.DataFrame(records)
outcome_columns = ['session_swing', 'exit_velo_mph_x', 'bat_speed_mph_contact_x',
                   'blast_bat_speed_mph_x', 'attack_angle_contact_x']
all_features = all_features.merge(poi[outcome_columns], on='session_swing', how='left', validate='many_to_one')
all_features = all_features.merge(hittrax[['session_swing', 'pitch', 'vertical_distance',
                                          'horizontal_distance', 'poi_z']],
                                  on='session_swing', how='left', validate='many_to_one')
all_features['session_key'] = all_features.user + '_' + all_features.session
results = all_features[all_features.cutoff_hz.eq(PRIMARY_CUTOFF_HZ)].copy()
all_features.to_csv(OUT / 'swing_metrics_all_filters.csv', index=False)
results.to_csv(OUT / 'swing_metrics.csv', index=False)
print(f'Retained {len(results)} swings from {results.user.nunique()} athletes at {PRIMARY_CUTOFF_HZ} Hz')
display(results[['plant_max_disp_cm', 'plant_vertical_range_cm', 'late100_max_disp_cm',
                 'exit_velo_mph_x', 'bat_speed_mph_contact_x']].describe().round(2))

In [ ]:
def within_session_model(table, outcome, predictors, label, bootstraps=BOOTSTRAPS):
    needed = ['user', 'session_key', outcome] + predictors
    d = table[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    counts = d.groupby('session_key')[outcome].transform('size')
    d = d[counts >= MIN_SWINGS].copy()
    if d.user.nunique() < 15 or len(d) < 50:
        return dict(model=label, status='insufficient complete sessions/athletes', n_swings=len(d),
                    n_athletes=d.user.nunique())
    centered = d[[outcome] + predictors] - d.groupby('session_key')[[outcome] + predictors].transform('mean')
    x, y = centered[predictors].to_numpy(), centered[outcome].to_numpy()
    if np.linalg.matrix_rank(x) < x.shape[1]:
        return dict(model=label, status='rank-deficient within-session predictors')
    fit = sm.OLS(y, x).fit(cov_type='cluster',
                         cov_kwds={'groups': d.user.to_numpy(), 'use_correction': False})
    residual_df = len(d) - d.session_key.nunique() - len(predictors)
    if residual_df <= 0:
        return dict(model=label, status='insufficient residual degrees of freedom')
    cluster_count = d.user.nunique()
    correction = cluster_count / (cluster_count - 1) * (len(d) - 1) / residual_df
    head_se = np.sqrt(fit.cov_params()[0, 0] * correction)
    critical = student_t.ppf(.975, cluster_count - 1)
    cluster_interval = fit.params[0] + np.array([-1, 1]) * critical * head_se
    cluster_p = 2 * student_t.sf(abs(fit.params[0] / head_se), cluster_count - 1) if head_se > 0 else np.nan
    clusters = [np.flatnonzero(d.user.to_numpy() == u) for u in d.user.unique()]
    rng = np.random.default_rng(SEED)
    estimates = []
    for _ in range(bootstraps):
        ix = np.concatenate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])
        if np.linalg.matrix_rank(x[ix]) == x.shape[1]:
            estimates.append(np.linalg.lstsq(x[ix], y[ix], rcond=None)[0][0])
    interval = np.quantile(estimates, [.025, .975]) if estimates else [np.nan, np.nan]
    return dict(model=label, status='estimated', outcome=outcome, predictor=predictors[0],
                controls=', '.join(predictors[1:]), n_swings=len(d), n_sessions=d.session_key.nunique(),
                n_athletes=d.user.nunique(), coefficient=float(fit.params[0]),
                cluster_ci_low=float(cluster_interval[0]), cluster_ci_high=float(cluster_interval[1]),
                bootstrap_ci_low=interval[0], bootstrap_ci_high=interval[1],
                cluster_p=float(cluster_p), bootstrap_success=len(estimates))

PRIMARY_PREDICTOR = 'plant_max_disp_pct_height'
BAT = 'bat_speed_mph_contact_x'
model_rows = []
for cutoff in CUTOFFS_HZ:
    table = all_features[all_features.cutoff_hz.eq(cutoff)]
    specifications = [
        ('EV: head + bat speed', 'exit_velo_mph_x', [PRIMARY_PREDICTOR, BAT]),
        ('EV: head + bat + pitch/location', 'exit_velo_mph_x',
         [PRIMARY_PREDICTOR, BAT, 'pitch', 'vertical_distance', 'horizontal_distance']),
        ('Bat speed: head movement', BAT, [PRIMARY_PREDICTOR]),
        ('EV: late vertical movement + bat', 'exit_velo_mph_x',
         ['late100_vertical_range_pct_height', BAT]),
    ]
    for label, outcome, predictors in specifications:
        row = within_session_model(table, outcome, predictors, label)
        row['cutoff_hz'] = cutoff
        row['role'] = 'primary' if label == 'EV: head + bat speed' and cutoff == PRIMARY_CUTOFF_HZ else 'exploratory/sensitivity'
        model_rows.append(row)
models = pd.DataFrame(model_rows)
models.to_csv(OUT / 'within_session_models.csv', index=False)
display(models[models.cutoff_hz.eq(PRIMARY_CUTOFF_HZ)].round(3))
print('Coefficients are per one percentage point of body height in movement.')
print('Models compare swings within the same athlete-session; uncertainty clusters by athlete.')
print('Pitch/location adjustment is a sensitivity analysis using a smaller complete-case sample.')
print('Other outcome/filter comparisons are exploratory; individual p-values are unadjusted.')

In [ ]:
def summarize_sessions(table, minimum):
    summaries = []
    for key, g in table.groupby('session_key'):
        if len(g) < minimum:
            continue
        cube_cm = np.stack([curves[s] for s in g.session_swing])
        cube_pct = cube_cm / g.height_m.to_numpy()[:, None, None]
        centered = cube_pct - cube_pct.mean(axis=0)
        rms_pct = np.sqrt(np.sum(centered ** 2) / ((len(g) - 1) * cube_pct.shape[1]))
        contacts = cube_pct[:, -1]
        contact_disp = np.sqrt(np.var(contacts, axis=0, ddof=1).sum())
        summaries.append(dict(session_key=key, user=g.user.iloc[0], session=g.session.iloc[0],
                              n_swings=len(g), trajectory_rms_pct_height=rms_pct,
                              contact_disp_pct_height=contact_disp,
                              mean_head_disp_cm=g.plant_max_disp_cm.mean(),
                              mean_bat_speed_mph=g[BAT].mean(), mean_ev_mph=g.exit_velo_mph_x.mean(),
                              ev_sd_mph=g.exit_velo_mph_x.std(), n_ev=g.exit_velo_mph_x.count(),
                              contact_depth_sd_in=g.poi_z.std(), n_depth=g.poi_z.count()))
    return pd.DataFrame(summaries, columns=['session_key', 'user', 'session', 'n_swings',
                        'trajectory_rms_pct_height', 'contact_disp_pct_height',
                        'mean_head_disp_cm', 'mean_bat_speed_mph', 'mean_ev_mph',
                        'ev_sd_mph', 'n_ev', 'contact_depth_sd_in', 'n_depth'])

def clustered_spearman(table, x, y, outcome_count, minimum):
    d = table.dropna(subset=[x, y]).copy()
    d = d[d[outcome_count] >= minimum]
    if len(d) < 15 or d.user.nunique() < 15:
        return dict(outcome=y, min_swings=minimum, status='insufficient sessions')
    rho = spearmanr(d[x], d[y]).statistic
    clusters = [g for _, g in d.groupby('user')]
    rng = np.random.default_rng(SEED)
    draws = []
    for _ in range(BOOTSTRAPS):
        sample = pd.concat([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))], ignore_index=True)
        value = spearmanr(sample[x], sample[y]).statistic
        if np.isfinite(value):
            draws.append(value)
    lo, hi = np.quantile(draws, [.025, .975])
    return dict(outcome=y, min_swings=minimum, status='estimated', n_sessions=len(d),
                n_athletes=d.user.nunique(), rho=rho, bootstrap_ci_low=lo, bootstrap_ci_high=hi)

sessions = summarize_sessions(results, MIN_SWINGS)
if sessions.empty:
    raise ValueError('No sessions have enough retained swings for repeatability estimates.')
sessions.to_csv(OUT / 'session_summary.csv', index=False)
associations = []
for minimum in (4, 5, 6):
    summary = summarize_sessions(results, minimum)
    for outcome, count in [('ev_sd_mph', 'n_ev'), ('contact_depth_sd_in', 'n_depth')]:
        row = clustered_spearman(summary, 'trajectory_rms_pct_height', outcome, count, minimum)
        row['role'] = 'secondary' if minimum == MIN_SWINGS and outcome == 'ev_sd_mph' else 'exploratory/sensitivity'
        associations.append(row)
associations = pd.DataFrame(associations)
associations.to_csv(OUT / 'repeatability_associations.csv', index=False)
display(sessions.head().round(3))
display(associations.round(3))
print('Repeatability describes dispersion around the session mean displacement trajectory.')
print("Contact dispersion is relative to each swing's foot-plant head position, not absolute lab position.")
print('Contact-depth variability also reflects pitch location; it is an exploratory timing proxy.')

In [ ]:
selected_session = str(SELECTED_SESSION) if SELECTED_SESSION is not None else sessions.sort_values(
    ['n_swings', 'session_key'], ascending=[False, True]).session_key.iloc[0]
g = results[results.session_key.eq(selected_session)]
if g.empty:
    raise KeyError(f'Unknown retained session {selected_session}')
cube = np.stack([curves[s] for s in g.session_swing])
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True)
phase = np.linspace(0, 100, 101)
for a, label in enumerate(['Forward/backward', 'Lateral (handedness aligned)', 'Vertical']):
    for curve in cube:
        axes[a].plot(phase, curve[:, a], alpha=.35, linewidth=1)
    axes[a].plot(phase, cube[:, :, a].mean(axis=0), color='black', linewidth=2, label='Session mean')
    axes[a].set(xlabel='Foot plant → contact (%)', ylabel='Displacement from foot plant (cm)', title=label)
    axes[a].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[0].legend()
fig.suptitle(f'Head movement repeatability: athlete-session {selected_session}, {len(g)} swings')
fig.tight_layout()
fig.savefig(FIG / 'session_trajectories.png', dpi=180, bbox_inches='tight')
plt.show()

selected = g.session_swing.iloc[0]
row = g.set_index('session_swing').loc[selected]
trace = traces[selected]
fig, ax = plt.subplots(figsize=(9, 4))
for a, label in enumerate(['Forward', 'Lateral', 'Vertical']):
    ax.plot(trace['time_ms'], trace['displacement_cm'][:, a], label=label)
ax.axvline(0, color='black', linestyle=':', label='Foot plant')
ax.axvline(row.plant_to_contact_ms, color='gray', linestyle='--', label='Contact')
ax.set(xlim=(-100, row.plant_to_contact_ms), xlabel='Time from foot plant (ms)',
       ylabel='Head displacement (cm)', title=f'Swing {selected}: four-marker centroid')
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig(FIG / 'example_swing.png', dpi=180, bbox_inches='tight')
plt.show()

plot_data = results[['session_key', PRIMARY_PREDICTOR, 'exit_velo_mph_x', BAT]].dropna()
plot_data = plot_data[plot_data.groupby('session_key').session_key.transform('size') >= MIN_SWINGS]
centered = plot_data[[PRIMARY_PREDICTOR, 'exit_velo_mph_x', BAT]] - plot_data.groupby('session_key')[
    [PRIMARY_PREDICTOR, 'exit_velo_mph_x', BAT]].transform('mean')
x = centered[PRIMARY_PREDICTOR].to_numpy()
y = centered.exit_velo_mph_x.to_numpy()
bat = centered[BAT].to_numpy()[:, None]
if len(centered) >= 50:
    x = x - bat @ np.linalg.lstsq(bat, x, rcond=None)[0]
    y = y - bat @ np.linalg.lstsq(bat, y, rcond=None)[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(x, y, alpha=.35, s=18)
    primary = models[(models.role == 'primary') & (models.status == 'estimated')]
    if not primary.empty:
        model = primary.iloc[0]
        grid = np.linspace(x.min(), x.max(), 100)
        axes[0].plot(grid, grid * model.coefficient, color='black', label='Conditional association')
        bounds = np.stack([grid * model.bootstrap_ci_low, grid * model.bootstrap_ci_high])
        axes[0].fill_between(grid, bounds.min(axis=0), bounds.max(axis=0), alpha=.15, color='black',
                             label='Bootstrap 95% slope interval')
        axes[0].legend(fontsize=8)
    axes[0].set(xlabel='Head displacement residual (% height)', ylabel='Exit velocity residual (mph)',
                title='Within session, accounting for bat speed')
    valid = sessions[(sessions.n_ev >= MIN_SWINGS)].dropna(subset=['trajectory_rms_pct_height', 'ev_sd_mph'])
    axes[1].scatter(valid.trajectory_rms_pct_height, valid.ev_sd_mph, alpha=.7)
    axes[1].set(xlabel='Trajectory variability (% height)', ylabel='Exit velocity SD (mph)',
                title='Session repeatability and EV variability')
    fig.tight_layout()
    fig.savefig(FIG / 'outcome_associations.png', dpi=180, bbox_inches='tight')
    plt.show()

In [ ]:
sensitivity_rows = []
primary_metrics = results.set_index('session_swing')
for cutoff in CUTOFFS_HZ:
    alternate = all_features[all_features.cutoff_hz.eq(cutoff)].set_index('session_swing')
    common = primary_metrics.index.intersection(alternate.index)
    for metric in ['plant_max_disp_cm', 'late100_vertical_range_cm']:
        a, b = primary_metrics.loc[common, metric], alternate.loc[common, metric]
        sensitivity_rows.append(dict(cutoff_hz=cutoff, metric=metric, common_swings=len(common),
                                     rank_correlation=spearmanr(a, b).statistic,
                                     median_abs_difference_cm=np.median(np.abs(a - b))))
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUT / 'filter_sensitivity.csv', index=False)
display(sensitivity.round(4))
manifest = dict(dataset='OpenBiomechanics Dataset v1', seed=SEED, bootstraps=BOOTSTRAPS,
                cutoff_hz=PRIMARY_CUTOFF_HZ, cutoff_sensitivity_hz=list(CUTOFFS_HZ),
                max_gap_frames=MAX_GAP_FRAMES, match_rmse_limit_m=MATCH_RMSE_LIMIT_M,
                head_pair_change_limit_m=HEAD_PAIR_CHANGE_LIMIT_M, min_session_swings=MIN_SWINGS,
                archive_sha256=ARCHIVES, csv_sha256={name: file_sha256(RAW / name)
                    for name in ['metadata.csv', 'poi_metrics.csv', 'hittrax.csv', 'landmarks.csv']},
                retained_swings=len(results), retained_athletes=int(results.user.nunique()),
                limitations=['Observational associations; no causal mechanical prescription.',
                             'Four-marker centroid is a head translation proxy; no gaze measurement.',
                             'Machine-pitch assessment; not a game-performance validation.',
                             'Exit velocity conditional on bat speed is not direct contact efficiency.',
                             'Session SD estimates use few swings; interpret intervals and sensitivity checks.',
                             'Secondary comparisons are exploratory and p-values are unadjusted.'])
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
primary = models[(models.role == 'primary') & (models.status == 'estimated')]
if not primary.empty:
    r = primary.iloc[0]
    print(f'Primary within-session association: {r.coefficient:+.2f} mph per 1% height of head displacement; '
          f'athlete-bootstrap 95% interval [{r.bootstrap_ci_low:+.2f}, {r.bootstrap_ci_high:+.2f}].')
    if r.bootstrap_ci_low <= 0 <= r.bootstrap_ci_high:
        print('The interval includes zero: this sample does not establish a clear direction for the association.')
print('A larger movement can still be highly repeatable. Less head movement is not assumed to be better.')
print(f'CSV results: {OUT}')
print(f'Figures: {FIG}')